In [ ]:
# 01_eda.ipynb — Study 1 EDA
# Run in Colab. Loads study1_daily.csv via Kaggle API.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# Credentials from Colab Secrets (key icon, left sidebar) -- never hardcode a real
# key here, this repo is public and it would stay in git history even if removed later.
import os
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

!kaggle datasets download -d halcyonvector/india-power-grid-nldc-daily-psp-reports -p data --unzip
df = pd.read_csv("data/study1_daily.csv", parse_dates=["date"])
print(df.shape)  # expect (2660, 144)
print(df["date"].min(), df["date"].max())

# --- Confirm target column ---
target_candidates = [
    "max_demand_met_total_mw",
    "evening_peak_demand_total_mw",
    "energy_met_total_mu",
]
for c in target_candidates:
    print(c, c in df.columns)

# --- Missingness ---
null_pct = df.isna().mean().sort_values(ascending=False) * 100
print(null_pct.head(20))

# --- Demand over time (gradient: color reflects each day's own demand value, not
#     just position -- a thin colored scatter over a faint connecting line) ---
plt.figure(figsize=(12, 4))
plt.plot(df["date"], df["max_demand_met_total_mw"], color="#4a3aa7", alpha=0.25, linewidth=1)
sc = plt.scatter(df["date"], df["max_demand_met_total_mw"], c=df["max_demand_met_total_mw"], cmap="cool", s=6)
plt.colorbar(sc, label="MW")
plt.title("National max demand met (MW) over time")
plt.show()

# --- Weekly/yearly seasonality (gradient bars: color = that bar's own value) ---
df["dow"] = df["date"].dt.dayofweek
df["year"] = df["date"].dt.year

plt.figure(figsize=(10, 4))
dow_avg = df.groupby("dow")["max_demand_met_total_mw"].mean()
norm = mpl.colors.Normalize(vmin=dow_avg.min(), vmax=dow_avg.max())
plt.bar(dow_avg.index, dow_avg.values, color=mpl.colormaps["PuBu"](norm(dow_avg.values)))
plt.title("Avg demand by day of week")
plt.show()

plt.figure(figsize=(10, 4))
year_avg = df.groupby("year")["max_demand_met_total_mw"].mean()
norm = mpl.colors.Normalize(vmin=year_avg.min(), vmax=year_avg.max())
plt.bar(year_avg.index.astype(str), year_avg.values, color=mpl.colormaps["RdPu"](norm(year_avg.values)))
plt.title("Avg demand by year")
plt.show()

# --- RES share trend (needed for Era 1 later) -- violet gradient, distinct from the
#     blue/magenta demand chart above ---
plt.figure(figsize=(12, 4))
plt.plot(df["date"], df["share_res_pct"], color="#4a3aa7", alpha=0.25, linewidth=1)
sc = plt.scatter(df["date"], df["share_res_pct"], c=df["share_res_pct"], cmap="Purples", s=6)
plt.colorbar(sc, label="RES share %")
plt.title("RES share % over time")
plt.show()

# --- Known gaps: cross-check against the known-gaps list from Phase 0 ---
full_range = pd.date_range(df["date"].min(), df["date"].max())
missing_dates = full_range.difference(df["date"])
print(f"{len(missing_dates)} missing dates (expect ~69 known gaps -- see Pipeline/known_gaps.json)")
